# Explore and virtualize AWS data

## Example of reading a single granule

As elsewhere, we define a workaround for configuring Obstore S3Stores on instances in AWS.

In [1]:
def make_store(url: str, region: str = "us-west-2"): 
    """Return an obstore FsspecStore for use with xarray / h5py / satpy."""
    from obstore.store import S3Store
    import boto3
    import os

    # 1. Hide empty environment variables from obstore's Rust backend
    for bad_var in ["AWS_WEB_IDENTITY_TOKEN_FILE", "AWS_ROLE_ARN"]:
        if os.environ.get(bad_var) == "":
            del os.environ[bad_var]

    # 2. Use boto3 to grab your working credentials (handles profiles, SSO, ~/.aws, etc.)
    session = boto3.Session(region_name=region)
    creds = session.get_credentials()

    config = {"region": region}

    # 3. If boto3 found credentials, extract the raw strings
    if creds:
        frozen = creds.get_frozen_credentials()
        config["access_key_id"] = frozen.access_key
        config["secret_access_key"] = frozen.secret_key
        if frozen.token:
            config["session_token"] = frozen.token

    # 4. Filter out any empty values so obstore doesn't crash on 'None'
    clean_config = {k: v for k, v in config.items() if v}

    # 5. Initialize obstore with the working credentials
    return S3Store.from_url(url, **clean_config)

Our AWS data copy is actually a zipfile containing the NetCDF along with some metadata.
It's still possible to read the data without extracting, but we have to use some workarounds.

In [2]:
import zipfile
import io
import xarray as xr
import obstore
from obstore.store import S3Store

S3_BUCKET = "airborne-smce-prod-user-bucket"
S3_REGION = "us-west-2"

key = ("JOIN/AWS/2026/01/24/"
       "W_NO-KSAT-Tromso,SAT,AWS1-MWR-1B-RAD_C_OHB__20260124121256"
       "_G_O_20260124102804_20260124120437_C_N____.nc")

store = make_store(f"s3://{S3_BUCKET}/", region=S3_REGION)
raw = bytes(obstore.get(store, key).bytes())

# The .nc file is a ZIP archive — extract the inner NetCDF
with zipfile.ZipFile(io.BytesIO(raw)) as zf:
    nc_name = [n for n in zf.namelist() if n.endswith(".nc")][0]
    nc_bytes = zf.read(nc_name)

ds_cal = xr.open_dataset(io.BytesIO(nc_bytes), group="data/calibration")
ds_geo = xr.open_dataset(io.BytesIO(nc_bytes), group="data/navigation")

In [3]:
ds_geo

<xarray.Dataset> Size: 185MB
Dimensions:                       (n_scans: 4866, n_fovs: 145, n_geo_groups: 4,
                                   n_fovs_cold: 25)
Dimensions without coordinates: n_scans, n_fovs, n_geo_groups, n_fovs_cold
Data variables: (12/17)
    time_startscan_utc_earthview  (n_scans) datetime64[ns] 39kB ...
    aws_lat                       (n_scans, n_fovs, n_geo_groups) float64 23MB ...
    aws_lon                       (n_scans, n_fovs, n_geo_groups) float64 23MB ...
    aws_solar_zenith_angle        (n_scans, n_fovs, n_geo_groups) float64 23MB ...
    aws_solar_azimuth_angle       (n_scans, n_fovs, n_geo_groups) float64 23MB ...
    orbit_angle                   (n_scans) float64 39kB ...
    ...                            ...
    aws_yawang                    (n_scans) float64 39kB ...
    aws_pitchang                  (n_scans) float64 39kB ...
    aws_moon_angles               (n_scans, n_fovs_cold, n_geo_groups) float64 4MB ...
    aws_sun_angles                (n_scans, n_fovs, n_geo_groups) float64 23MB ...
    aws_surface_type              (n_scans, n_fovs, n_geo_groups) float32 11MB ...
    aws_terrain_elevation         (n_scans, n_fovs, n_geo_groups) float32 11MB ...

Now, we do some basic visualizations of the granule to understand its structure.

In [4]:
from matplotlib import pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [6]:
ds_geo_gg0 = ds_geo.isel(n_geo_groups = 0)
proj = ccrs.Robinson()

In [ ]:
fig, ax = plt.subplots(subplot_kw={"projection": proj})

scatter = ax.scatter(
    ds_geo_gg0["aws_lon"],
    ds_geo_gg0["aws_lat"],
    c = ds_cal["aws_toa_brightness_temperature"].isel(n_channels=0),
    cmap="viridis",
    marker=".",
    transform = ccrs.PlateCarree()
)
ax.add_feature(cfeature.LAND, facecolor="lightgrey", zorder=0)
plt.colorbar(scatter, label="TOA brightness temp.")
plt.show()